## Python-dotenv (API key management)

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv

# Paths are relative to the repo root - run this notebook from there.
load_dotenv('./scripts/.env')

# Test: get the API key from the environment variable
os.environ.get('OPENAI_API_KEY')

## Create PubMed Query based on intervention and contraindication and check if all relevant articles are found

In [ ]:
# Load in data
reports = pd.read_excel('Overzicht_CI_rapporten_HealthBase.xlsx', sheet_name=1)
reports['CI-rapport nummer '] = reports['CI-rapport nummer '].ffill()
reports['CI-rapport nummer '] = pd.to_numeric(reports['CI-rapport nummer '], errors='coerce').fillna(0).astype(int)


In [ ]:
# Extract CI and I terms
terms = reports.dropna(subset=['PICO CI '])
terms = terms.copy()
terms.loc[:, 'CI-rapport nummer '] = terms['CI-rapport nummer '].astype(int, errors='ignore')
terms = terms.loc[:,['CI-rapport nummer ', 'PICO CI ', 'PICO interventie ', 'Startdatum', 'datum search ']]
# Rename the columns
terms.columns = ['rapport_number', 'contraindication', 'intervention', 'date_start_search', 'date_end_search']


In [ ]:
article_cases = reports[['CI-rapport nummer ', 'PICO CI ', 'PICO interventie ', 'PMID geïncludeerde studies ']].copy()
article_cases.columns = ['rapport_number', 'contraindication', 'intervention', 'PMID']
article_cases['contraindication'] = article_cases['contraindication'].ffill()
article_cases['intervention'] = article_cases['intervention'].ffill()
article_cases


In [ ]:
terms

## Define Functions

#### Functions to run all queries and return dataframe of results

In [ ]:
import os
import subprocess
import time
import pandas as pd

from dotenv import load_dotenv

load_dotenv("./.env")
os.environ["NCBI_API_KEY"] = os.environ.get("PUBMED_API_KEY")

from metapub import PubMedFetcher

fetch = PubMedFetcher()

######################################################################################################################################
######################## Helper functions: ###########################################################################################
######################################################################################################################################


def add_tiab(df: pd.DataFrame) -> pd.DataFrame:
    """
    function that takes datframe with a title and abstract column and adds a column with title + abstract concatenated
    """
    # If dataframe is empty, return the same empty dataframe
    if df.empty:
        return df

    # add a column with title + abstract concatenated
    df["TiAb"] = df.apply(
        lambda row: row["title"]
        if pd.isna(row["abstract"])
        else row["title"] + " " + row["abstract"],
        axis=1,
    )

    # Put 'TiAb' column in front
    cols = ["TiAb"] + [col for col in df.columns if col != "TiAb"]
    df = df[cols]

    return df


def nbib_to_dict(nbib_entry: str) -> dict:
    """Convert an nbib entry to a dictionary."""
    lines = nbib_entry.strip().split("\n")
    result_dict = {}
    current_key = None
    current_value = ""
    known_keys = [
        "PMID",
        "OWN",
        "STAT",
        "DCOM",
        "LR",
        "IS",
        "VI",
        "IP",
        "DP",
        "TI",
        "PG",
        "LID",
        "AB",
        "FAU",
        "AU",
        "AD",
        "AUID",
        "LA",
        "PT",
        "DEP",
        "PL",
        "TA",
        "JT",
        "JID",
        "PMC",
        "OTO",
        "OT",
        "COIS",
        "EDAT",
        "MHDA",
        "CRDT",
        "PHST",
        "AID",
        "PST",
        "SO",
    ]
    reading_multiline = (
        False  # State variable to track if we are reading a multiline field
    )

    for line in lines:
        # If we encounter a line that contains "- "
        potential_key = line.split("- ", 1)[0].strip()
        if potential_key in known_keys:
            if current_key:  # Save the previous key-value pair
                if current_key == "AU":
                    if "AU" in result_dict:
                        result_dict["AU"].append(current_value.strip())
                    else:
                        result_dict["AU"] = [current_value.strip()]
                else:
                    result_dict[current_key] = current_value.strip()

            current_key, current_value = line.split("- ", 1)
            current_key = current_key.strip()
            reading_multiline = current_key in ["AB", "TI"]  # Set the state variable
        else:
            # Handle multiline fields by appending to the current value
            if reading_multiline:
                current_value += " " + line.strip()

    # Save the last key-value pair
    if current_key:
        if current_key == "AU":
            if "AU" in result_dict:
                result_dict["AU"].append(current_value.strip())
            else:
                result_dict["AU"] = [current_value.strip()]
        else:
            result_dict[current_key] = current_value.strip()

    return result_dict


def convert_to_dataframe(list_of_dicts: list) -> pd.DataFrame:
    # Extract relevant data from the list of dictionaries
    data = [
        {
            "pubmed_id": item.get("PMID", ""),
            "title": item.get("TI", ""),
            "publication_date": item.get("EDAT", ""),
            "year": item.get("DP", ""),
            "journal": item.get("JT", ""),
            "abstract": item.get("AB", ""),
            "authors": ", ".join(item.get("AU", [])),
        }
        for item in list_of_dicts
    ]

    # Convert the list of dictionaries to a pandas DataFrame
    df = pd.DataFrame(data)

    # Convert all columns to strings
    df = df.astype(str)

    # Replace "nan" strings with an empty string
    df = df.replace("nan", "")

    # Remove trailing/leading whitespaces and explicit double quotations from Title, Journal, Abstract, Authors
    clean = ["title", "journal", "abstract", "authors", "year"]
    for column in clean:
        if column in df.columns:  # Check if the column exists
            df[column] = df[column].str.strip().str.replace('"', "")

    # Keep only unique results in the final dataframe
    unique_results = df.drop_duplicates(subset="pubmed_id", keep="first")

    # add TiAb and replace NaN with empty string if needed
    final_results = add_tiab(unique_results)
    if not final_results.empty:
        final_results["TiAb"] = final_results["TiAb"].fillna("")

    return final_results


############################################################################################################################
##################### Main Function ########################################################################################
############################################################################################################################


def get_articles(query: str, max_results=9999) -> pd.DataFrame:
    """Returns a dataframe with articles found by the query"""

    # Get pmids by query
    pmids = fetch.pmids_for_query(query, retmax=max_results)

    # Get nbibs
    # retrieve al nbib files for each article pbased on PMID and add as column to dataframe
    # use batchsize = 500 and pause 1s after to comply with api rules
    nbibs = []
    batchsize = 500

    for i in range(0, len(pmids), batchsize):
        ids = " ,".join(pmids[i : i + batchsize])
        nbib = subprocess.check_output(
            f"efetch -db pubmed -id {ids} -format medline",
            shell=True,
            text=True,
        )
        nbib = nbib.split(sep="\n\n")
        for id in nbib:
            nbibs.append(id)
        time.sleep(1)

    print(
        f"Great, number of nbibs ({len(nbibs)}) is matching the number of articles ({len(pmids)})"
    ) if (len(nbibs) == len(pmids)) else print(
        f"WARNING: Number of nbibs ({len(nbibs)}) not matching number of articles ({len(pmids)})"
    )

    # Convert nbib to dictionary
    article_dicts = [nbib_to_dict(entry) for entry in nbibs]

    # Convert list of dictionaries to dataframe for further use in pipeline
    articles_df = convert_to_dataframe(article_dicts)

    # Add nbib column
    articles_df["nbib"] = nbibs

    return articles_df

#### Function that creates a PubMed query from a description of contraindication and intervention

In [ ]:
# Function that takes CI and I term and creates a query
from langchain.chat_models import ChatOpenAI
from langchain import PromptTemplate
from langchain.chains import LLMChain


def make_query(ci_term: str, iv_term: str) -> pd.DataFrame:

    """takes as input contraindication term and intervention term, returns a dataframe of results from PubMed using a chatGPT created query"""

    intervention = iv_term
    contraindication = ci_term

    # create a chatGPT model
    llm = ChatOpenAI(model_name='gpt-4', temperature=0.2, max_tokens=3000, openai_api_key=os.environ.get('OPENAI_API_KEY'))

    # create a prompt template
    template = '''
    Make a pubmed query that is sensitive enough to find all relevant papers that have information on using {intervention} in patients with {contraindication}. 
    When forming the PubMed query based on these intervention and contraindication terms, strictly adhere to the following guidelines:
    1 Translate {intervention} and {contraindication} to English if they appear to contain non-English language. 
    2 Only extract exact medical terms to include in the query. Be precise in the selection of terms, using specific medical terminology (e.g., use 'angioedema' instead of 'patients with history of angioedma').
    3 Avoid including broad or colloquial expressions in the query. Exclude non-medical terms and general descriptors, such as 'history' and 'patients'.
    4 If a term contains multiple distinct medical components, separate them and include each one individually in the query using the logical OR operator. For example, instead of using (Neprilysin inhibitors: sacubitril), break it down into individual components: ((neprilysin inhibitors) OR sacubitril).
    5 Ensure that each term used in the query represents an exact medical concept or entity, without any additional context or descriptors.
    6 Include any relevant synonyms or alternative spellings for the medical terms, but only if they are exact matches within the medical domain. Limit yourself in the number of synonyms used, and use only exact medical synonyms. For example, if 'angioedema' is a term, include 'Quincke's edema' and 'angio-edema' in the PubMed query, but do not include 'urticaria' or 'hives' because they are not exact medical synonyms of 'angioedema'.
    7 If a term includes an abbreviation, always include both the abbreviation and the full-spelled-out version in the query. For example, use ((DPP-4 Inhibitor) OR (dipeptidyl peptidase-4 inhibitor)) instead of only (DPP-4 inhibitor).
    8 Never use double quotations around any term in the query.
    9 Make sure the parenthesis seperate OR and AND operators adequately: (Angioedema) AND ((sacubitril) OR (neprilysin inhibitor)) instead of (Angioedema AND sacubitril OR neprilysin inhibitor).
    10 Keep the query as short as possible, focusing solely on the medical terms for interventions and contraindications, but not shorter than needed for the query to be sensitive enough to find all relevant articles. 
    11 Match opening parenthesis with corresponding clossing parentheses. Always check that all opening parentheses have a corresponding closing parentheses, and vice versa.
    Finally, answer by only returning the PubMed query, don't say anything else.
    '''

    # create a prompt
    prompt = PromptTemplate(
        input_variables = ['intervention', 'contraindication'],
        template = template
        )

    # create a chain and run it
    chain = LLMChain(llm=llm, prompt=prompt)
    output = chain.run({'intervention': intervention, 'contraindication': contraindication})
    print(output) # print the output

    return output


## Run Functions:
#### Iterate over dataframe of contraindications and interventions and make dataframe with query results

In [ ]:
terms.iloc[249:n,:]

In [ ]:
# Return a list of queries based on the CI and I terms for the first n rows of the dataframe
n = 332
queries = []

for index, row in terms.iloc[:n,:].iterrows():
    query = make_query(row['contraindication'], row['intervention'])
    queries.append(query)

    

In [ ]:
len(queries)


In [ ]:
# Remove "patient with ", "patients with ", "Patient with ", "Patients with "
queries = [query.replace("patient with ", "").replace("patients with ", "").replace("Patient with ", "").replace("Patients with ", "") for query in queries]

# Remove " patients", " patient", " Patients", " Patient"
queries = [query.replace(" patients", "").replace(" patient", "").replace(" Patients", "").replace(" Patient", "") for query in queries]


In [ ]:
def extract_year(date, end_date=False):
    if pd.notnull(date):
        if end_date:
            return str(date.year+1)
        else:
            return str(date.year)
    else:
        if end_date:
            return str(2023)
        else:
            return str(1900)

terms['query_dated'] = terms.apply(lambda row: f'{row["query"]} AND (("{extract_year(row["date_start_search"])}"[Date - Publication] : "{extract_year(row["date_end_search"], end_date=True)}"[Date - Publication]))', axis=1)
terms

In [ ]:
# save to file
terms.to_csv('all_queries.csv')


In [ ]:
df_list = []
# Iterate over each row in the terms dataframe
for index, row in terms.iloc[300:,:].iterrows():
    # Get articles for the current query
    results = get_articles(row['query_dated'], max_results=1000)

    # Insert rapport_number, contraindication, and intervention columns to the results
    results.insert(0, 'rapport_number', row['rapport_number'])
    results.insert(1, 'contraindication', row['contraindication'])
    results.insert(2, 'intervention', row['intervention'])

    # Append the results to the df_list
    df_list.append(results)

# Concatenate all dataframes in the df_list to create the final dataframe
final_df = pd.concat(df_list)


In [ ]:
# save to file
final_df.to_csv('all_articles_pre-inclusions.csv')


In [ ]:
final_df = pd.read_csv('all_articles_pre-inclusions.csv', index_col=0)


In [ ]:
final_df.head(3)

In [ ]:
# Export dataframe to csv
# results_with_tiab.to_csv('./scripts/pubmed_output/chad_query_results_with_tiab.csv', index=False)

## Test number of found PMIDs that were included

In [ ]:
reports.head(3)

In [ ]:
pmid_included = list(reports['PMID geïncludeerde studies '].dropna().drop_duplicates(keep='first'))

In [ ]:
final_df['case'] = final_df['pubmed_id'].apply(lambda x: 1 if x in pmid_included else 0)


In [ ]:
final_df

In [ ]:
final_df = final_df.drop(columns=['nbib'])
cols = final_df.columns.tolist()
cols.insert(0, cols.pop(cols.index('case')))
final_df = final_df.reindex(columns= cols)
final_df

In [ ]:
final_df['case'].value_counts()[1]


In [ ]:
# save to file
# final_df.to_csv('all_articles_CI_unbalanced.csv')
final_df = pd.read_csv('all_articles_CI_unbalanced.csv', index_col=0)


In [ ]:
final_df

In [ ]:
pmid_missed = [pmid for pmid in pmid_included if pmid not in final_df['pubmed_id'].tolist()]
pmid_missed = [str(pmid) for pmid in pmid_missed]

In [ ]:
# get data from all pmid missed
article_cases['PMID'] = article_cases['PMID'].astype(str)
article_cases_missed = article_cases[article_cases['PMID'].isin(pmid_missed)]
# remove duplicate rows based on PMID
article_cases_missed = article_cases_missed.drop_duplicates(subset='PMID')


In [ ]:
article_cases_missed

In [ ]:
from tqdm import tqdm

df_list = []
# Iterate over each row in the article_cases_missed dataframe
for index, row in tqdm(article_cases_missed.iloc[:,:].iterrows(), total=article_cases_missed.shape[0]):
    # Get articles for the current query
    results = get_articles(row['PMID'], max_results=1000)

    # Insert rapport_number, contraindication, and intervention columns to the results
    results.insert(0, 'rapport_number', row['rapport_number'])
    results.insert(1, 'contraindication', row['contraindication'])
    results.insert(2, 'intervention', row['intervention'])

    # Append the results to the df_list
    df_list.append(results)

# Concatenate all dataframes in the df_list to create the final dataframe
final_df_missed = pd.concat(df_list)

In [ ]:
final_df_missed['case'] = 1
cols = list(final_df_missed.columns)
cols = [cols[-1]] + cols[:-1]
final_df_missed = final_df_missed[cols]
final_df_missed.drop(columns=['nbib'], inplace=True)
final_df_missed

In [ ]:
final_data = pd.concat([final_df, final_df_missed]).drop_duplicates(subset='pubmed_id').sort_values(by='rapport_number')
final_data

In [ ]:
final_data['case'].value_counts()

In [ ]:
# save to file
final_data.to_csv('data_unbalanced_72K.csv', index=False)
